# Spacetime Error Visualizations for Surface-Code Circuits

Visualizes **when** (which round) and **what type** of errors occur across a
rotated surface-code memory experiment, using Stim's spacetime / detector
diagrams plus the interactive Crumble editor.

The circuit is built with the **same noise channels** as the trained dataset
(`Stim_generate_datasets.ipynb`):
- `before_round_data_depolarization=p`
- `after_reset_flip_probability=p`
- `after_clifford_depolarization=p`

Trained datasets use `ROUNDS=2`; here we default to `ROUNDS=5` so the spacetime
picture is richer, while keeping the same circuit family.

Each diagram is also written to `plots/*.html` so it can be opened in a browser
and screenshotted for slides. Run with `uv run jupyter ...` (TF 2.15 / Stim env).

In [ ]:
# Cell 1 — Imports + build one circuit
import os
from circuit_generators import get_builtin_circuit

# Visualization config. Higher rounds than the trained ROUNDS=2 gives a richer
# spacetime picture; the 3 noise channels match the real dataset circuit family.
VIZ_D, VIZ_P, VIZ_ROUNDS = 5, 0.010, 5

circuit = get_builtin_circuit(
    'surface_code:rotated_memory_z',
    distance=VIZ_D, rounds=VIZ_ROUNDS,
    before_round_data_depolarization=VIZ_P,
    after_reset_flip_probability=VIZ_P,
    after_clifford_depolarization=VIZ_P,
)

os.makedirs('plots', exist_ok=True)
print(circuit.num_ticks, 'ticks')

## 1. Spacetime error graph (interactive 3D)

Built from the detector error model (DEM). Each node is a detector; edges are
error mechanisms that flip a pair of detectors. The vertical axis is time
(rounds), so this shows the full **spacetime** matching structure. Drag to rotate.

In [ ]:
# Cell 2 — Spacetime error graph via the DEM.
# decompose_errors=True splits correlated errors into graphlike edges; if a
# circuit can't be decomposed, fall back to the undecomposed DEM.
try:
    dem = circuit.detector_error_model(decompose_errors=True)
except Exception as e:
    print(f'decompose_errors=True failed ({e}); falling back to undecomposed DEM')
    dem = circuit.detector_error_model()

matchgraph = dem.diagram('matchgraph-3d-html')
matchgraph

**How to read this.** Every **node** is a detector — a stabilizer measurement whose value flipped between rounds, signalling something nearby went wrong. Every **edge** is a single physical error mechanism, connecting the pair of detectors that error lights up. The **vertical axis is time**: an edge running upward is an error visible across consecutive rounds (e.g. a measurement error), while edges within a layer connect neighbouring qubits in one round (a data error). Boundary nodes at the sides absorb errors that flip only one detector.

This is exactly the graph MWPM (PyMatching) matches on: given the lit-up detectors, it finds the most likely set of edges explaining them. So this figure *is* the decoding problem — and the structure your quantized NN decoder learns to solve implicitly from the same syndrome bits. More rounds or higher `p` make the graph taller/denser and the matching harder.

## 2. Per-round detector slices (the "timestamps")

One slice per tick: which detectors are defined at that layer of the circuit.
Reading left-to-right walks through time, round by round.

In [ ]:
# Cell 3 — Per-round detector slices across every tick.
detslices = circuit.diagram('detslice-svg', tick=range(0, circuit.num_ticks))
detslices

**How to read this.** Each small panel is a single **tick** (circuit time step), shown left-to-right as a timeline. Coloured regions mark the detectors defined at that moment — red/blue correspond to the two stabilizer types (X- and Z-checks). Walking across the strip replays one full experiment: reset → repeated stabilizer rounds → final data readout.

This is the **syndrome being built up over time** — one panel per measurement layer. A single sample in your training set is essentially the sequence of these detector values flattened into the input vector. Use this slide to show *what one dataset example looks like as it is measured round by round*, before any decoder sees it.

## 3. Operations + detectors overlaid ("what happened that layer")

Same per-tick slices, but with the circuit operations (gates, resets,
measurements) drawn on top of the detector regions, so you can see what each
layer actually does.

In [ ]:
# Cell 4 — Operations overlaid on detector slices.
detslice_ops = circuit.diagram('detslice-with-ops-svg', tick=range(0, circuit.num_ticks))
detslice_ops

**How to read this.** Same per-tick slices as above, but now the actual **circuit operations** are drawn on top: resets (R), two-qubit CX gates (lines between qubits), and measurements (M). So each panel answers *what the hardware physically does* in that layer to produce the detectors, not just which detectors exist.

This is the view to use when explaining **how the syndrome is generated** — the gate schedule that entangles data and ancilla qubits and reads out the stabilizers each round. Pair it with the previous slide: that one shows the syndrome data, this one shows the circuit that creates it.

## 4. Crumble (interactive editor)

Stim's interactive circuit editor. Navigate layers with **Q** / **E**, and
click qubits to inject Paulis and watch them propagate through the circuit.

In [ ]:
# Cell 5 — Crumble interactive editor.
crumble = circuit.diagram('interactive')
crumble

**How to read this.** Crumble is Stim's interactive playground (opens from the saved `crumble.html`). Use **Q** / **E** to step backward/forward through circuit layers, and click a qubit to inject a Pauli error (X / Y / Z). The editor then **propagates** that error forward and highlights which detectors it eventually trips.

This makes the abstract error model concrete: drop one X error mid-circuit and literally watch it spread and light up a detector pair — the same pair that shows up as an edge in the 3D matchgraph above. Best used as a **live demo** for 'what does a single error actually do?'; for static slides the 3D matchgraph and detector slices screenshot better.

## Save standalone HTML files for slides

Each diagram is written to its own file under `plots/` (gitignored). Open in a
browser and screenshot for slides.

In [ ]:
# Cell 6 — Persist each diagram to its own HTML/SVG-in-HTML file.
os.makedirs('plots', exist_ok=True)
for name, helper in [
    ('matchgraph',   matchgraph),
    ('detslices',    detslices),
    ('detslice_ops', detslice_ops),
    ('crumble',      crumble),
]:
    path = f'plots/{name}.html'
    with open(path, 'w') as f:
        f.write(str(helper))
    print(f'wrote {path}')